# H1398 HC04_A11 Breakpoint Explorer

Scan all 3882 windows on H1398_rep1 HC04_A11 for internal breakpoints.

Two algorithms are compared:
- **detect_internal_breakpoint()**: Quadrant asymmetry + diagonal profile kink detection
- **detect_breakpoint_by_nan_density()**: 5x5 sliding window NaN density detection

Results:
- `results/h1398_a11_breakpoint_qc.tsv` (detect_internal_breakpoint): 1183 abnormal windows
- `results/h1398_a11_nan_density_qc.tsv` (detect_breakpoint_by_nan_density): 0 abnormal windows

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py

sys.path.insert(0, "/share/home/xhhuang24/project/PanTE/background_pool/scripts")
from detect_internal_breakpoint import detect_internal_breakpoint, cleanup_matrix
from detect_breakpoint_by_nan_density import detect_breakpoint_by_nan_density

warnings.filterwarnings("ignore")

WORKDIR = "/share/home/xhhuang24/project/PanTE/background_pool"
OUTDIR = "results"
os.chdir(WORKDIR)

SAMPLE = "H1398_rep1"
CHROM = "HC04_A11"
BIN_SIZE = 4000

In [ ]:
# Load breakpoint QC results from both algorithms

# 1. detect_internal_breakpoint() results
qc_path = os.path.join(WORKDIR, OUTDIR, "h1398_a11_breakpoint_qc.tsv")
df_asym = pd.read_csv(qc_path, sep="\t")

print("=" * 60)
print("detect_internal_breakpoint() (quadrant asymmetry + profile kink)")
print("=" * 60)
print(f"Total windows: {len(df_asym)}")
print(f"Abnormal windows: {df_asym['is_abnormal'].sum()}")
print(f"PASS rate: {(~df_asym['is_abnormal']).mean():.1%}")
print()
print("Top 20 abnormal windows by asymmetry:")
display_cols = ["window_id", "window_start", "window_end", "best_K", "best_K_bp", "asymmetry", "contrast"]
print(df_asym[df_asym["is_abnormal"] == True].sort_values("asymmetry", ascending=False)[display_cols].head(20).to_string())

print()
print("=" * 60)
print("detect_breakpoint_by_nan_density() (5x5 NaN sliding window)")
print("=" * 60)

nan_qc_path = os.path.join(WORKDIR, OUTDIR, "h1398_a11_nan_density_qc.tsv")
df_nan = pd.read_csv(nan_qc_path, sep="\t")
print(f"Total windows: {len(df_nan)}")
print(f"Abnormal windows: {df_nan['is_abnormal'].sum()}")
ok_mask = df_nan["status"] == "ok"
if ok_mask.sum() > 0:
    print(f"PASS rate: {(~df_nan.loc[ok_mask, 'is_abnormal']).mean():.1%}")

# Show distribution of upper_zero_frac
print()
print("upper_zero_frac statistics (ok status):")
print(df_nan.loc[ok_mask, "upper_zero_frac"].describe())

# Show top windows by upper_zero_frac
print()
print("Top 20 windows by upper_zero_frac:")
print(df_nan.sort_values("upper_zero_frac", ascending=False)[["window_id", "window_start", "window_end", "best_K", "upper_zero_frac", "is_abnormal"]].head(20).to_string())

In [ ]:
def plot_nan_density_breakpoint(sample, chrom, window_id, bin_size=4000, figsize=(14, 5)):
    """
    Plot heatmap with NaN-density breakpoint line overlaid.

    Uses detect_breakpoint_by_nan_density() to find breakpoint and visualizes
    the 5x5 sliding window NaN density scan results.

    Parameters
    ----------
    sample   : str  e.g. "H1398_rep1"
    chrom    : str  e.g. "HC04_A11"
    window_id: int or str  e.g. 975 or "w000975"
    bin_size : int  bp per bin (default 4000)
    figsize  : tuple
    """
    if isinstance(window_id, int):
        window_id = f"w{window_id:06d}"

    # Load HDF5
    h5_path = os.path.join(WORKDIR, OUTDIR, "windows", sample, f"{sample}__{chrom}_ln_oe.h5")
    with h5py.File(h5_path, "r") as f:
        mat = f[window_id]["matrix"][:]
        window_start = int(f[window_id]["window_start"][()])
        window_end = int(f[window_id]["window_end"][()])

    # Detect breakpoint using NaN density
    result = detect_breakpoint_by_nan_density(mat, bin_size=bin_size, window_start=window_start)

    # Prepare matrix for plotting
    X = cleanup_matrix(mat, symmetrize=True)
    np.fill_diagonal(X, np.nan)

    # Create figure
    fig, axes = plt.subplots(1, 2, figsize=figsize, gridspec_kw={"width_ratios": [3, 1]})

    # --- Left: heatmap ---
    ax = axes[0]
    im = ax.imshow(X, cmap="RdBu_r", aspect="auto", origin="upper",
                   vmin=-2, vmax=2)
    ax.set_title(f"{sample} {chrom} {window_id}\n"
                 f"{window_start:,}–{window_end:,} bp | "
                 f"is_abnormal={result['is_abnormal']} | "
                 f"best_K={result['best_K']} ({result['best_K_bp']:,} bp)\n"
                 f"upper_zero_frac={result['upper_zero_frac']:.4f}")
    ax.set_xlabel("Bin index")
    ax.set_ylabel("Bin index")

    # Overlay breakpoint line at bin K-0.5
    if result["status"] == "ok" and result["best_K"] >= 0:
        bp_k = result["best_K"]
        ax.axhline(bp_k - 0.5, color="lime", linewidth=2, linestyle="--",
                   label=f"breakpoint K={bp_k}")
        ax.axvline(bp_k - 0.5, color="lime", linewidth=2, linestyle="--")
        ax.legend(loc="upper right")

    plt.colorbar(im, ax=ax, label="ln(O/E)", shrink=0.8)

    # --- Right: NaN density profile ---
    ax2 = axes[1]
    n = X.shape[0]
    # Compute 5x5 NaN density scan
    nan_fracs = []
    for k in range(n - 4):
        window = X[k:k + 5, k:k + 5]
        nan_count = np.isnan(window).sum()
        nan_fracs.append(nan_count / 25.0)
    nan_fracs = np.array(nan_fracs)

    k_vals = np.arange(len(nan_fracs))
    x_vals = (k_vals + 2) * bin_size / 1e3  # center of 5x5 window in kb

    ax2.barh(x_vals, nan_fracs, height=bin_size/1e3 * 0.8, color="coral", alpha=0.7)
    ax2.set_xlabel("5x5 NaN fraction")
    ax2.set_ylabel("Position (kb)")
    ax2.set_title("NaN density profile")
    ax2.set_xlim(0, max(0.1, nan_fracs.max() * 1.2))

    if result["status"] == "ok" and result["best_K"] >= 2:
        bp_k = result["best_K"]
        bp_y = bp_k * bin_size / 1e3
        ax2.axhline(bp_y, color="lime", linewidth=2, linestyle="--")
        ax2.text(nan_fracs.max() * 0.5, bp_y, f"K={bp_k}", color="lime", fontsize=10, va="center")

    ax2.invert_yaxis()
    plt.tight_layout()
    plt.show()

    # Print result
    print(result)
    return result

In [ ]:
# ── Interactive cell ───────────────────────────────────────────────────────────
# Change window_id and re-run this cell to explore different windows
#
# Examples:
#   window_id = 0           # first window of A11
#   window_id = 77         # high asymmetry (0.67)
#   window_id = 84         # high asymmetry (0.57)
#   window_id = 975        # HC04_D03 reference window
#   window_id = 2328       # highest asymmetry window

window_id = 77  # <- change this and re-run

# Quick lookup from both algorithms
print("=" * 60)
print("detect_internal_breakpoint() result:")
print("=" * 60)
row_asym = df_asym[df_asym["window_id"] == window_id]
if len(row_asym) == 0:
    print(f"Window {window_id} not found in asymmetry results")
else:
    r = row_asym.iloc[0]
    print(f"window_id={window_id} | {r['window_start']:,}–{r['window_end']:,} bp")
    print(f"is_abnormal={r['is_abnormal']} | best_K={r['best_K']} ({r['best_K_bp']:,} bp) | asym={r['asymmetry']:.3f} | contrast={r['contrast']:.2f}")

print()
print("=" * 60)
print("detect_breakpoint_by_nan_density() result:")
print("=" * 60)
row_nan = df_nan[df_nan["window_id"] == window_id]
if len(row_nan) == 0:
    print(f"Window {window_id} not found in NaN density results")
else:
    r = row_nan.iloc[0]
    print(f"window_id={window_id} | {r['window_start']:,}–{r['window_end']:,} bp")
    print(f"is_abnormal={r['is_abnormal']} | best_K={r['best_K']} ({r['best_K_bp']:,} bp) | upper_zero_frac={r['upper_zero_frac']:.4f} | status={r['status']}")

# Plot NaN density visualization
print()
print("=" * 60)
print("NaN Density Visualization:")
print("=" * 60)
plot_nan_density_breakpoint(SAMPLE, CHROM, window_id, bin_size=BIN_SIZE)